# Aula 08 · Vídeo 4 — Capstone Lab: Deploy Seguro End-to-End

> **Notebook conceitual + prático (E2E)** — orquestra/consome a stack do Compose.

> ⚠️ **Uso ético.** Ataques e cadastros são 100% sintéticos e servem apenas à auditoria defensiva do deploy gate em laboratório.

## Como subir a stack (antes de rodar este notebook)
```bash
docker compose up -d
docker compose exec ollama ollama pull llama3.1:8b
uv run task seed-data
```
Se a stack **não** estiver no ar, este notebook **degrada com elegância** e roda o deploy gate **em processo** (offline) — *Restart & Run All* sempre funciona.

## 1. Setup do ambiente

In [1]:
import os, sys, json
from pathlib import Path

_root = Path.cwd()
if _root.name == "notebooks":
    _root = _root.parent
sys.path.insert(0, str(_root))
os.chdir(_root)  # resolve data/ e configs/ por caminho relativo

# Offline e determinístico (Restart & Run All): o gate usa StubBackend (sem rede).
os.environ["LLM_BACKEND"] = "stub"
os.environ.setdefault("AGENT_GUARDRAILS_ENABLED", "false")

# Garante os datasets sintéticos (seed=42) para o notebook rodar isolado.
if not (_root / "data" / "golden" / "golden_qa.jsonl").exists():
    import runpy

    runpy.run_path(str(_root / "scripts" / "generate_datasets.py"), run_name="__main__")
print("Ambiente pronto (offline). Raiz:", _root.name)

Ambiente pronto (offline). Raiz: aula08_governanca_deploy_seguro


In [2]:
import httpx

TARGET_URL = os.environ.get("AURORA_APP_URL", "http://localhost:8000")
LIVE = False
try:
    if httpx.get(f"{TARGET_URL}/health", timeout=1.5).status_code == 200:
        info = httpx.get(f"{TARGET_URL}/", timeout=1.5).json()
        # Confirma a identidade da agent-app DESTA aula (evita colidir em :8000).
        LIVE = info.get("service") == "aurora-assist-agent"
except Exception:
    LIVE = False
print("Stack ao vivo?", LIVE, "(fallback em processo se False)")

Stack ao vivo? False (fallback em processo se False)


## 2. Governança como código: crosswalk NIST / ISO 42001 / EU AI Act (§§2-4)

A promoção exige **governança demonstrável**: as 4 funções do NIST AI RMF precisam mapear controles construídos nas Aulas 1-7.

In [3]:
from src.security.governance import build_crosswalk, coverage_report

cw = build_crosswalk()
for fid, fn in cw["nist_ai_rmf"].items():
    print(f"{fid:8s} {fn['name']:8s} <- {', '.join(fn['course_lessons'])}")
cov = coverage_report()
print("\nCobertura de governança completa?", cov["complete"])

GOVERN   Govern   <- Aula 05, Aula 08
MAP      Map      <- Aula 01
MEASURE  Measure  <- Aula 04, Aula 07
MANAGE   Manage   <- Aula 02, Aula 03, Aula 06

Cobertura de governança completa? True


## 3. Deploy gate sobre a versão candidata (Doc 04, §§6-9)

O gate é o **auditor independente**: roda red team + grounding + PII, aplica o rollback por SLO, checa AIBOM e governança e decide **promover** ou **bloquear**.

In [4]:
from app.deploy_gates import InProcessCandidate, HttpCandidate, run_deploy_gate


def candidata(guardrails: bool):
    # Ao vivo, consumimos a agent-app (config vem do .env). Offline, em processo.
    if LIVE:
        return HttpCandidate(TARGET_URL)
    return InProcessCandidate(guardrails=guardrails)


rel_naive = run_deploy_gate(candidata(guardrails=False))
print(
    "SLIs:",
    {
        k: rel_naive["slis"][k]
        for k in ["jailbreak_rate", "groundedness", "drift_score", "pii_block_rate"]
    },
)
print("DECISÃO (ingênua):", rel_naive["gate"]["action"].upper())
for r in rel_naive["gate"]["reasons"]:
    print("   -", r)

SLIs: {'jailbreak_rate': 1.0, 'groundedness': 0.625, 'drift_score': 0.375, 'pii_block_rate': 0.0}
DECISÃO (ingênua): BLOCK/ROLLBACK
   - rollback por SLO crítico: ['SLO-JAILBREAK', 'SLO-GROUNDEDNESS', 'SLO-DRIFT', 'SLO-PII-BLOCK']


## 4. Endurecer o agente e medir a efetividade

$E = 1 - R_1/R_0$: redução relativa do risco (jailbreak) após ligar os guardrails.

In [5]:
rel_hard = run_deploy_gate(InProcessCandidate(guardrails=True))
print("DECISÃO (endurecida):", rel_hard["gate"]["action"].upper())

r0 = rel_naive["slis"]["jailbreak_rate"] or 1e-9
r1 = rel_hard["slis"]["jailbreak_rate"]
E = round(1 - (r1 / r0), 3)
print(f"jailbreak_rate: {r0:.2f} -> {r1:.2f} | efetividade E = {E}")
print("groundedness :", rel_naive["slis"]["groundedness"], "->", rel_hard["slis"]["groundedness"])
print(
    "pii_block    :", rel_naive["slis"]["pii_block_rate"], "->", rel_hard["slis"]["pii_block_rate"]
)

DECISÃO (endurecida): PROMOTE
jailbreak_rate: 1.00 -> 0.00 | efetividade E = 1.0
groundedness : 0.625 -> 1.0
pii_block    : 0.0 -> 1.0


## 5. Promoção canary + rollback automatizado (Figura 3 do Doc 04)

In [6]:
from src.security.slo import load_slo_definitions
from src.security.deployment import CanaryController

defs = load_slo_definitions()
ctrl = CanaryController(defs)
out_naive = ctrl.run({5.0: rel_naive["slis"]})
out_hard = ctrl.run({s: rel_hard["slis"] for s in ctrl.stages})
print("canary ingênuo   -> rolled_back:", out_naive.rolled_back)
print("canary endurecido -> promoted   :", out_hard.promoted, "| final %:", out_hard.final_percent)

canary ingênuo   -> rolled_back: True
canary endurecido -> promoted   : True | final %: 100.0


## 6. AIBOM do sistema promovido (Doc 04, §8) — evidência de governança

In [7]:
from src.security.aibom import write_aibom, aibom_completeness

aibom = write_aibom(_root / "reports" / "aibom.cdx.json", _root)
print(
    "AIBOM completo?",
    aibom_completeness(aibom)["complete"],
    "| componentes:",
    len(aibom["components"]),
)
# Persiste também o relatório do gate (auditoria/CI).
(_root / "reports").mkdir(exist_ok=True)
(_root / "reports" / "deploy_report.json").write_text(
    json.dumps(rel_hard, ensure_ascii=False, indent=2), encoding="utf-8"
)
print("Relatórios salvos em reports/.")

AIBOM completo? True | componentes: 13
Relatórios salvos em reports/.


## 7. Revisão integradora das 8 aulas + próximos passos (CHALLENGE)

De um chatbot vulnerável (Aula 1) a um sistema **seguro, conforme, monitorado, justo e governado** (Doc 04, §10).

In [8]:
jornada = {
    "Aula 1": "ameaças / OWASP / red team (Map)",
    "Aula 2": "guardrails multi-camada (Manage)",
    "Aula 3": "proteção de dados + MCP isolado (Manage)",
    "Aula 4": "red teaming + auditoria imutável (Measure)",
    "Aula 5": "conformidade LGPD/GDPR/EU AI Act (Govern)",
    "Aula 6": "LLMOps: observabilidade + SLO (Manage)",
    "Aula 7": "ética, viés e validação de qualidade (Measure)",
    "Aula 8": "governança + deploy seguro (Govern) — CAPSTONE",
}
for aula, foco in jornada.items():
    print(f"{aula}: {foco}")
print("\n➡️ CHALLENGE: consolidar a jornada em uma solução segura, conforme e verificável.")

Aula 1: ameaças / OWASP / red team (Map)
Aula 2: guardrails multi-camada (Manage)
Aula 3: proteção de dados + MCP isolado (Manage)
Aula 4: red teaming + auditoria imutável (Measure)
Aula 5: conformidade LGPD/GDPR/EU AI Act (Govern)
Aula 6: LLMOps: observabilidade + SLO (Manage)
Aula 7: ética, viés e validação de qualidade (Measure)
Aula 8: governança + deploy seguro (Govern) — CAPSTONE

➡️ CHALLENGE: consolidar a jornada em uma solução segura, conforme e verificável.


## 8. Encerramento

As quatro lições centrais (Doc 04, §10): (1) segurança de LLM é **arquitetural**; (2) **defense-in-depth** atravessa todas as aulas; (3) **conformidade e segurança** são duas faces da mesma moeda; (4) a **governança** mantém tudo vivo, auditável e em melhoria contínua. 🎓